# P1-E03 CAUEEG Recording Linkage, Duplicate Resolution and Provenance Lock

**Paper 1, Experiment 3 — consume the P1-E02. official no-overlap cohort**

## Summary

This notebook begins only from `13_LOCKED_NORMAL_MCI_AD_OFFICIAL_NO_OVERLAP_COHORT.csv` (1,045 recordings), verifies its SHA-256 against the P1-E02.2 run record, links every retained recording to its EDF and event JSON, reconciles the P1-E01 header audit, audits exact duplicates, and writes a recording-level provenance manifest.

`serial` is a recording identifier, not a verified participant identifier. Official train, validation, and test assignments are carried forward unchanged. Any exact duplicate component spanning official splits or diagnostic labels stops execution for manual review. Same-label duplicates within one split may be reduced deterministically without modifying raw files.
**.


## Context & Methods

### Purpose

P1-E02. established an official no-overlap Normal/MCI/AD cohort of 1,045 recordings. P1-E03 verifies that every locked record maps to the expected raw EDF and event JSON, detects missing or unreadable inputs, resolves safe within-split exact duplicates, and freezes file-level provenance for downstream experiments.

### Key assumptions

1. P1-E02. is the only permitted label/cohort source.
2. `serial` identifies a recording; biological participant identity is unavailable.
3. `dementia-no-overlap.json`, already preserved by P1-E02.2, supplies the available held-out overlap protection.
4. Official split assignments are immutable; this notebook never reallocates a record.
5. Exact duplicate EDFs are defined by identical SHA-256 hashes.
6. A duplicate component crossing diagnoses or official splits is a leakage/conflict risk and causes a hard stop.
7. Only a same-label, same-split duplicate component can retain one deterministic representative (lowest normalized serial).
8. Raw EDF and event files are never modified or deleted.
9. Age is carried forward without full-cohort matching, scaling, imputation, or adjustment.


## Data

### 1. Setup and fixed paths


In [ ]:
from pathlib import Path
import hashlib, json, re, sys, platform, warnings
import numpy as np
import pandas as pd
import matplotlib as mpl
import matplotlib.pyplot as plt
import seaborn as sns

DATA_ROOT = Path(r'D:\EEG_AD_Research\data\caueeg-dataset\caueeg-dataset')
PROJECT_ROOT = Path(r'D:\EEG_AD_Research')
E01_TABLES = PROJECT_ROOT/'paper1_experiments'/'paper1_outputs'/'p1_e01_raw_audit'/'tables'
E02_ROOT = PROJECT_ROOT/'paper1_experiments'/'paper1_outputs'/'p1_e02_label_harmonisation'
E02_TABLES = E02_ROOT/'tables'
OUTPUT_ROOT = PROJECT_ROOT/'paper1_experiments'/'paper1_outputs'/'p1_e03_recording_linkage_lock'
FIG_DIR, TABLE_DIR = OUTPUT_ROOT/'figures', OUTPUT_ROOT/'tables'
for directory in (OUTPUT_ROOT, FIG_DIR, TABLE_DIR):
    directory.mkdir(parents=True, exist_ok=True)

LOCKED_INPUT = E02_TABLES/'13_LOCKED_NORMAL_MCI_AD_OFFICIAL_NO_OVERLAP_COHORT.csv'
E02_RUN_RECORD = E02_ROOT/'run_record.json'
HEADER_AUDIT = E01_TABLES/'07_eeg_header_audit.csv'
CLASS_ORDER = ['Normal', 'MCI', 'AD']
SPLIT_ORDER = ['train', 'validation', 'test']
EXPECTED_COUNTS = pd.DataFrame(
    {'Normal':[367,34,35], 'MCI':[334,28,33], 'AD':[189,10,15]},
    index=SPLIT_ORDER,
)
CLASS_COLORS = {'Normal':'#0072B2', 'MCI':'#E69F00', 'AD':'#D55E00'}

mpl.rcParams.update({
    'font.family':'Arial', 'font.size':9, 'axes.titlesize':10,
    'axes.labelsize':9, 'legend.fontsize':8, 'savefig.dpi':600,
    'axes.spines.top':False, 'axes.spines.right':False,
    'pdf.fonttype':42, 'ps.fonttype':42,
})
sns.set_theme(style='ticks', context='paper')

def save_table(frame, name, index=False):
    frame.to_csv(TABLE_DIR/f'{name}.csv', index=index)
    try:
        frame.to_excel(TABLE_DIR/f'{name}.xlsx', index=index)
    except Exception as exc:
        warnings.warn(f'Excel export skipped ({exc}); CSV was saved.')

def save_figure(fig, name, width_mm=183, height_mm=100):
    fig.set_size_inches(width_mm/25.4, height_mm/25.4)
    for extension in ('png', 'pdf', 'svg'):
        fig.savefig(FIG_DIR/f'{name}.{extension}', bbox_inches='tight', facecolor='white')

def normalize_serial(value):
    text = str(value).strip()
    if not text or text.lower() in {'nan', 'none'}:
        return ''
    if re.fullmatch(r'\d+\.0', text):
        text = text[:-2]
    digits = ''.join(re.findall(r'\d+', text))
    return digits.zfill(5) if digits else text

def sha256_file(path, block_size=4*1024*1024):
    digest = hashlib.sha256()
    with Path(path).open('rb') as handle:
        for block in iter(lambda: handle.read(block_size), b''):
            digest.update(block)
    return digest.hexdigest()

required_inputs = {
    'P1-E02.2 locked cohort': LOCKED_INPUT,
    'P1-E02.2 run record': E02_RUN_RECORD,
    'P1-E01 EEG header audit': HEADER_AUDIT,
}
missing = [f'{name}: {path}' for name, path in required_inputs.items() if not path.exists()]
if missing:
    raise FileNotFoundError(
        'Required prior outputs are missing. Run the corrected P1-E01 and P1-E02.2 first:\n'
        + '\n'.join(missing)
    )
assert DATA_ROOT.exists(), f'Dataset root not found: {DATA_ROOT}'
print('Locked cohort:', LOCKED_INPUT)
print('Header audit:', HEADER_AUDIT)
print('Output root:', OUTPUT_ROOT)


### 2. Load and authenticate the P1-E02.2 locked cohort

The input must contain exactly the 1,045 official records and reproduce the audited class-by-split counts. The manifest hash must match the value recorded by P1-E02.2.


In [ ]:
with E02_RUN_RECORD.open('r', encoding='utf-8') as handle:
    e02_run = json.load(handle)

expected_input_hash = e02_run.get('locked_manifest_sha256', '')
observed_input_hash = sha256_file(LOCKED_INPUT)
if not expected_input_hash:
    raise ValueError('P1-E02.2 run_record.json does not contain locked_manifest_sha256.')
if observed_input_hash != expected_input_hash:
    raise AssertionError(
        'P1-E02.2 locked-manifest SHA-256 mismatch. Do not continue with an altered or stale cohort file.'
    )

cohort = pd.read_csv(LOCKED_INPUT, dtype={'serial':str, 'identity_group':str})
required_columns = {
    'serial', 'age', 'official_split', 'experimental_label',
    'experimental_label_code', 'analysis_unit', 'identity_group',
    'participant_identity_verified', 'split_source', 'tokens_text', 'label_reason'
}
missing_columns = required_columns - set(cohort.columns)
if missing_columns:
    raise ValueError(f'P1-E02.2 locked cohort lacks columns: {sorted(missing_columns)}')

cohort['serial'] = cohort.serial.map(normalize_serial)
cohort['identity_group'] = cohort.identity_group.map(normalize_serial)
if cohort.serial.eq('').any():
    raise ValueError('Blank serials in the P1-E02.2 locked cohort.')
if cohort.serial.duplicated().any():
    display(cohort[cohort.serial.duplicated(False)].sort_values('serial'))
    raise ValueError('Duplicate recording serials in the P1-E02.2 locked cohort.')

cohort['official_split'] = cohort.official_split.astype(str).str.strip().str.lower()
cohort['experimental_label'] = cohort.experimental_label.astype(str).str.strip()
observed_counts = (
    cohort.groupby(['official_split','experimental_label']).size()
    .unstack(fill_value=0)
    .reindex(index=SPLIT_ORDER, columns=CLASS_ORDER, fill_value=0)
    .astype(int)
)

input_checks = pd.DataFrame([
    {'check':'locked-manifest SHA-256 matches P1-E02.2 run record', 'pass':observed_input_hash==expected_input_hash, 'detail':observed_input_hash},
    {'check':'exactly 1,045 locked recordings loaded', 'pass':len(cohort)==1045, 'detail':len(cohort)},
    {'check':'recording serials are unique and nonblank', 'pass':cohort.serial.ne('').all() and not cohort.serial.duplicated().any(), 'detail':cohort.serial.nunique()},
    {'check':'official splits are exactly train/validation/test', 'pass':set(cohort.official_split)==set(SPLIT_ORDER), 'detail':cohort.official_split.value_counts().to_dict()},
    {'check':'class-by-split counts reproduce P1-E02.2', 'pass':observed_counts.equals(EXPECTED_COUNTS), 'detail':observed_counts.to_dict()},
    {'check':'analysis unit remains recording', 'pass':cohort.analysis_unit.astype(str).str.lower().eq('recording').all(), 'detail':cohort.analysis_unit.value_counts().to_dict()},
    {'check':'participant identity is not claimed as verified', 'pass':~cohort.participant_identity_verified.astype(str).str.lower().isin(['true','1']).any(), 'detail':cohort.participant_identity_verified.value_counts(dropna=False).to_dict()},
    {'check':'identity group equals released recording serial', 'pass':cohort.identity_group.eq(cohort.serial).all(), 'detail':int((cohort.identity_group!=cohort.serial).sum())},
])
save_table(input_checks, '01_p1_e02_2_input_authentication_checks')
display(observed_counts)
display(input_checks)
if not input_checks['pass'].all():
    raise AssertionError('STOP — P1-E02.2 INPUT AUTHENTICATION FAILED')
print('P1-E02.2 INPUT AUTHENTICATED')


## Results

### 3. Link locked recordings to EDF, event JSON and the P1-E01 header audit


In [ ]:
cohort['recording_id'] = cohort.serial
cohort['edf_path'] = cohort.serial.map(lambda s: str(DATA_ROOT/'signal'/'edf'/f'{s}.edf'))
cohort['event_path'] = cohort.serial.map(lambda s: str(DATA_ROOT/'event'/f'{s}.json'))
cohort['edf_exists'] = cohort.edf_path.map(lambda p: Path(p).is_file())
cohort['event_exists'] = cohort.event_path.map(lambda p: Path(p).is_file())
cohort['edf_bytes'] = cohort.edf_path.map(lambda p: Path(p).stat().st_size if Path(p).is_file() else np.nan)
cohort['event_bytes'] = cohort.event_path.map(lambda p: Path(p).stat().st_size if Path(p).is_file() else np.nan)

headers = pd.read_csv(HEADER_AUDIT)
required_header_columns = {'relative_path','extension'}
if not required_header_columns.issubset(headers.columns):
    raise ValueError(f'P1-E01 header audit lacks columns: {sorted(required_header_columns-set(headers.columns))}')
headers['serial'] = headers.relative_path.astype(str).map(
    lambda value: normalize_serial(Path(value.replace('\\','/')).stem)
)
edf_headers = headers[headers.extension.astype(str).str.lower().eq('.edf')].copy()
relevant_headers = edf_headers[edf_headers.serial.isin(cohort.serial)].copy()
header_multiplicity = relevant_headers.groupby('serial').size()
ambiguous_header_serials = header_multiplicity[header_multiplicity.ne(1)]
if len(ambiguous_header_serials):
    display(ambiguous_header_serials.rename('header_rows').to_frame())
    raise ValueError('P1-E01 header audit has multiple EDF rows for a locked recording serial.')

header_columns = [
    c for c in ['serial','status','sfreq_Hz','n_channels','n_samples',
                'duration_seconds','channel_names','n_annotations']
    if c in relevant_headers.columns
]
linked = cohort.merge(relevant_headers[header_columns], on='serial', how='left', validate='one_to_one')
linked['header_reconciled'] = linked.serial.isin(relevant_headers.serial)
if 'status' in linked.columns:
    linked['header_readable'] = linked.status.astype(str).str.lower().eq('readable')
else:
    linked['header_readable'] = False
linked['linkage_ready'] = (
    linked.edf_exists & linked.event_exists &
    linked.header_reconciled & linked.header_readable
)

linkage_summary = (
    linked.groupby(['official_split','experimental_label']).agg(
        locked_records=('recording_id','size'),
        edf_missing=('edf_exists', lambda x: (~x).sum()),
        event_missing=('event_exists', lambda x: (~x).sum()),
        header_unreconciled=('header_reconciled', lambda x: (~x).sum()),
        header_unreadable=('header_readable', lambda x: (~x).sum()),
        linkage_ready=('linkage_ready','sum'),
    ).reset_index()
)
save_table(linked, '02_locked_recording_file_linkage_audit')
save_table(linkage_summary, '03_linkage_summary_by_split_and_class')
display(linkage_summary)


### 4. Audit event JSON readability

This is a structural readability audit only. Event timing and interval eligibility belong to P1-E04.


In [ ]:
def audit_event(path):
    record = {
        'event_json_readable':False, 'event_root_type':'',
        'event_count':np.nan, 'event_error':''
    }
    if not Path(path).is_file():
        record['event_error'] = 'missing'
        return record
    try:
        with Path(path).open('r', encoding='utf-8-sig') as handle:
            obj = json.load(handle)
        record.update(
            event_json_readable=True,
            event_root_type=type(obj).__name__,
            event_count=len(obj) if isinstance(obj,(list,dict)) else 1,
        )
    except Exception as exc:
        record['event_error'] = repr(exc)
    return record

event_audit = pd.DataFrame([
    {'recording_id':row.recording_id, **audit_event(row.event_path)}
    for row in linked.itertuples(index=False)
])
linked = linked.merge(event_audit, on='recording_id', how='left', validate='one_to_one')
save_table(event_audit, '04_event_json_readability_audit')
display(event_audit.event_json_readable.value_counts(dropna=False).rename('recordings').to_frame())


### 5. Detect exact duplicates and stop on label or split conflicts

Only equal-size EDFs can be byte-identical, so hashing is first restricted to repeated file sizes. Every confirmed duplicate component is checked for diagnostic-label and official-split consistency before any representative is selected.


In [ ]:
ready = linked[linked.linkage_ready & linked.event_json_readable].copy()
repeated_size = ready[ready.duplicated('edf_bytes', keep=False)].copy()
hash_cache = {}
for number, path in enumerate(repeated_size.edf_path.unique(), 1):
    hash_cache[path] = sha256_file(path)
    if number % 100 == 0:
        print(f'Hashed {number}/{repeated_size.edf_path.nunique()} repeated-size EDF files')

ready['edf_sha256_screen'] = ready.edf_path.map(hash_cache).fillna('not_hashed_unique_size')
duplicate_rows = ready[
    ready.edf_sha256_screen.ne('not_hashed_unique_size') &
    ready.duplicated('edf_sha256_screen', keep=False)
].copy()
duplicate_rows = duplicate_rows.sort_values(['edf_sha256_screen','recording_id'])
if len(duplicate_rows):
    duplicate_rows['duplicate_component'] = (
        duplicate_rows.groupby('edf_sha256_screen', sort=True).ngroup().add(1)
        .map(lambda x: f'DUP_{x:03d}')
    )
else:
    duplicate_rows['duplicate_component'] = pd.Series(dtype='object')

duplicate_summary = (
    duplicate_rows.groupby('duplicate_component').agg(
        members=('recording_id','size'),
        label_count=('experimental_label','nunique'),
        split_count=('official_split','nunique'),
        labels=('experimental_label', lambda x:' | '.join(sorted(set(x)))),
        splits=('official_split', lambda x:' | '.join(sorted(set(x)))),
        recording_ids=('recording_id', lambda x:' | '.join(x)),
    ).reset_index()
)
label_conflicts = duplicate_summary[duplicate_summary.label_count.gt(1)].copy()
cross_split_duplicates = duplicate_summary[duplicate_summary.split_count.gt(1)].copy()
save_table(duplicate_rows, '05_confirmed_exact_duplicate_recordings')
save_table(duplicate_summary, '06_duplicate_component_summary')
save_table(label_conflicts, '07_conflicting_label_duplicate_components')
save_table(cross_split_duplicates, '08_cross_split_duplicate_components')
display(duplicate_summary)

if len(label_conflicts):
    raise AssertionError('STOP — exact duplicate EDF components contain conflicting diagnostic labels. Review table 07.')
if len(cross_split_duplicates):
    raise AssertionError('STOP — exact duplicate EDF components cross official splits. Review table 08; do not auto-resolve.')
print('Duplicate conflict gates passed.')


### 6. Resolve only safe within-split duplicates and document all exclusions


In [ ]:
ready['duplicate_component'] = ''
ready['duplicate_representative'] = True
ready['exclusion_reason'] = ''

if len(duplicate_rows):
    component_map = duplicate_rows.set_index('recording_id').duplicate_component
    member_mask = ready.recording_id.isin(component_map.index)
    ready.loc[member_mask, 'duplicate_component'] = ready.loc[member_mask, 'recording_id'].map(component_map)
    representatives = duplicate_rows.groupby('duplicate_component').recording_id.min().to_dict()
    ready.loc[member_mask, 'duplicate_representative'] = ready.loc[member_mask].apply(
        lambda row: row.recording_id == representatives[row.duplicate_component], axis=1
    )
    ready.loc[member_mask & ~ready.duplicate_representative, 'exclusion_reason'] = 'exact_duplicate_same_label_same_split_nonrepresentative'

retained = ready[ready.duplicate_representative].copy().sort_values('recording_id').reset_index(drop=True)
excluded = linked[~linked.recording_id.isin(retained.recording_id)].copy()
safe_duplicate_exclusions = set(ready.loc[~ready.duplicate_representative, 'recording_id'])
excluded['exclusion_reason'] = np.select([
    ~excluded.edf_exists,
    ~excluded.event_exists,
    ~excluded.header_reconciled,
    ~excluded.header_readable,
    ~excluded.event_json_readable,
    excluded.recording_id.isin(safe_duplicate_exclusions),
], [
    'missing_edf', 'missing_event_json', 'unreconciled_edf_header',
    'unreadable_edf_header', 'unreadable_event_json',
    'exact_duplicate_same_label_same_split_nonrepresentative',
], default='other_linkage_exclusion')

save_table(excluded, '09_excluded_recordings_with_reasons')
exclusion_summary = (
    excluded.groupby(['official_split','experimental_label','exclusion_reason']).size()
    .rename('recordings').reset_index()
)
save_table(exclusion_summary, '10_exclusion_summary_by_split_class_reason')
display(exclusion_summary)
print('Input recordings:', len(linked), '| retained:', len(retained), '| excluded:', len(excluded))


### 7. Freeze the recording-level provenance manifest

All retained EDF and event files are hashed. The row hash includes the immutable official split and experimental label.


In [ ]:
for number, path in enumerate(retained.edf_path.unique(), 1):
    if path not in hash_cache:
        hash_cache[path] = sha256_file(path)
    if number % 200 == 0:
        print(f'Provenance-hashed {number}/{retained.edf_path.nunique()} retained EDF files')

retained['edf_sha256'] = retained.edf_path.map(hash_cache)
retained['event_sha256'] = retained.event_path.map(sha256_file)
retained['manifest_row_sha256'] = retained.apply(
    lambda row: hashlib.sha256(
        f'{row.recording_id}|{row.official_split}|{row.experimental_label}|{row.edf_sha256}|{row.event_sha256}'.encode('utf-8')
    ).hexdigest(), axis=1
)
retained['manifest_version'] = 'P1-E03-v2-corrected'

manifest_columns = [
    'recording_id','serial','age','official_split','experimental_label',
    'experimental_label_code','analysis_unit','identity_group',
    'participant_identity_verified','split_source','tokens_text','label_reason',
    'edf_path','event_path','edf_bytes','event_bytes','edf_sha256','event_sha256',
    'manifest_row_sha256','manifest_version','sfreq_Hz','n_channels','n_samples',
    'duration_seconds','channel_names','event_count','duplicate_component',
]
manifest_columns = [c for c in manifest_columns if c in retained.columns]
final_manifest = retained[manifest_columns].copy()
save_table(final_manifest, '11_LOCKED_RECORDING_LEVEL_PROVENANCE_MANIFEST')

retained_counts = (
    final_manifest.groupby(['official_split','experimental_label']).size()
    .unstack(fill_value=0).reindex(index=SPLIT_ORDER, columns=CLASS_ORDER, fill_value=0)
    .astype(int)
)
save_table(retained_counts.reset_index(), '12_retained_counts_by_official_split_and_class')
display(final_manifest.head())
display(retained_counts)


### 8. Publication figures


In [ ]:
before = cohort.experimental_label.value_counts().reindex(CLASS_ORDER, fill_value=0)
after = final_manifest.experimental_label.value_counts().reindex(CLASS_ORDER, fill_value=0)
plot_counts = pd.DataFrame({'P1-E02.2 locked input':before, 'P1-E03 retained':after})
save_table(plot_counts.reset_index(names='experimental_label'), '13_cohort_flow_class_counts')

fig, axes = plt.subplots(1,2)
x = np.arange(len(CLASS_ORDER)); width = .36
axes[0].bar(x-width/2, before.values, width, label='P1-E02.2 input', color='#A0A0A0')
axes[0].bar(x+width/2, after.values, width, label='P1-E03 retained', color=[CLASS_COLORS[c] for c in CLASS_ORDER])
axes[0].set(xticks=x, xticklabels=CLASS_ORDER, ylabel='Recordings', title='Recording retention after linkage and deduplication')
axes[0].legend(frameon=False)
for container in axes[0].containers:
    axes[0].bar_label(container, padding=2, fontsize=7)

sns.boxplot(
    data=final_manifest, x='experimental_label', y='duration_seconds',
    order=CLASS_ORDER, hue='experimental_label', palette=CLASS_COLORS,
    legend=False, ax=axes[1],
)
axes[1].set(xlabel='Diagnostic group', ylabel='Recording duration (seconds)', title='Raw duration in retained cohort')
save_figure(fig, 'figure_1_recording_retention_and_duration', 183, 82)
plt.show()

flow = pd.DataFrame([
    {'stage':'P1-E02.2 official locked cohort', 'recordings':len(cohort)},
    {'stage':'EDF + event + readable header', 'recordings':int((linked.linkage_ready & linked.event_json_readable).sum())},
    {'stage':'P1-E03 provenance-locked recordings', 'recordings':len(final_manifest)},
])
save_table(flow, '14_recording_flow')
fig, ax = plt.subplots()
bars = ax.barh(flow.stage, flow.recordings, color=['#9E9E9E','#56B4E9','#009E73'])
ax.invert_yaxis(); ax.bar_label(bars, padding=3)
ax.set(xlabel='Recordings', title='P1-E03 recording linkage and provenance flow')
save_figure(fig, 'figure_2_recording_linkage_flow', 165, 78)
plt.show()


### 9. Final integrity gates and reproducibility record


In [ ]:
post_counts_plus_exclusions = (
    pd.concat([
        final_manifest[['official_split','experimental_label','recording_id']],
        excluded[['official_split','experimental_label','recording_id']],
    ], ignore_index=True)
    .groupby(['official_split','experimental_label']).size()
    .unstack(fill_value=0).reindex(index=SPLIT_ORDER, columns=CLASS_ORDER, fill_value=0)
    .astype(int)
)

checks = pd.DataFrame([
    {'check':'authenticated P1-E02.2 input contains 1,045 recordings', 'pass':len(cohort)==1045, 'detail':len(cohort)},
    {'check':'input class-by-split counts reproduce P1-E02.2', 'pass':observed_counts.equals(EXPECTED_COUNTS), 'detail':observed_counts.to_dict()},
    {'check':'each input recording appears retained or excluded exactly once', 'pass':len(final_manifest)+len(excluded)==len(cohort) and set(final_manifest.recording_id).isdisjoint(set(excluded.recording_id)), 'detail':{'input':len(cohort),'retained':len(final_manifest),'excluded':len(excluded)}},
    {'check':'retained plus excluded reconcile to official input counts', 'pass':post_counts_plus_exclusions.equals(EXPECTED_COUNTS), 'detail':post_counts_plus_exclusions.to_dict()},
    {'check':'all retained EDF files exist', 'pass':final_manifest.edf_path.map(lambda p:Path(p).is_file()).all(), 'detail':int((~final_manifest.edf_path.map(lambda p:Path(p).is_file())).sum())},
    {'check':'all retained event files exist', 'pass':final_manifest.event_path.map(lambda p:Path(p).is_file()).all(), 'detail':int((~final_manifest.event_path.map(lambda p:Path(p).is_file())).sum())},
    {'check':'all retained EDF headers are reconciled and readable', 'pass':retained.header_reconciled.all() and retained.header_readable.all(), 'detail':int((~retained.header_reconciled | ~retained.header_readable).sum())},
    {'check':'all retained event JSON files are readable', 'pass':retained.event_json_readable.all(), 'detail':int((~retained.event_json_readable).sum())},
    {'check':'no conflicting-label duplicate components', 'pass':len(label_conflicts)==0, 'detail':len(label_conflicts)},
    {'check':'no cross-split duplicate components', 'pass':len(cross_split_duplicates)==0, 'detail':len(cross_split_duplicates)},
    {'check':'no exact duplicate EDF remains twice', 'pass':not final_manifest.edf_sha256.duplicated().any(), 'detail':int(final_manifest.edf_sha256.duplicated().sum())},
    {'check':'official split values are preserved without reallocation', 'pass':set(final_manifest.official_split).issubset(set(SPLIT_ORDER)), 'detail':final_manifest.official_split.value_counts().to_dict()},
    {'check':'analysis unit remains recording', 'pass':final_manifest.analysis_unit.astype(str).str.lower().eq('recording').all(), 'detail':final_manifest.analysis_unit.value_counts().to_dict()},
    {'check':'participant identity remains explicitly unverified', 'pass':~final_manifest.participant_identity_verified.astype(str).str.lower().isin(['true','1']).any(), 'detail':final_manifest.participant_identity_verified.value_counts(dropna=False).to_dict()},
    {'check':'row provenance hashes are unique', 'pass':not final_manifest.manifest_row_sha256.duplicated().any(), 'detail':int(final_manifest.manifest_row_sha256.duplicated().sum())},
])
save_table(checks, '15_final_integrity_checks')
display(checks)

status = (
    'P1-E03 PASSED — RECORDING-LEVEL PROVENANCE MANIFEST LOCKED'
    if checks['pass'].all()
    else 'STOP — P1-E03 CHECKS FAILED; REVIEW BEFORE INTERVAL SELECTION'
)
manifest_csv = TABLE_DIR/'11_LOCKED_RECORDING_LEVEL_PROVENANCE_MANIFEST.csv'
run_record = {
    'experiment':'P1-E03 corrected CAUEEG recording linkage, duplicate resolution and provenance lock',
    'timestamp_utc':pd.Timestamp.utcnow().isoformat(),
    'data_root':str(DATA_ROOT),
    'inputs':{
        'p1_e02_2_locked_manifest':str(LOCKED_INPUT),
        'p1_e02_2_locked_manifest_sha256':observed_input_hash,
        'p1_e01_header_audit':str(HEADER_AUDIT),
        'p1_e01_header_audit_sha256':sha256_file(HEADER_AUDIT),
    },
    'input_recordings':len(cohort),
    'retained_recordings':len(final_manifest),
    'excluded_recordings':len(excluded),
    'retained_class_counts':{str(k):int(v) for k,v in final_manifest.experimental_label.value_counts().items()},
    'retained_split_counts':{str(k):int(v) for k,v in final_manifest.official_split.value_counts().items()},
    'confirmed_duplicate_rows':len(duplicate_rows),
    'conflicting_label_duplicate_components':len(label_conflicts),
    'cross_split_duplicate_components':len(cross_split_duplicates),
    'identity_interpretation':{
        'analysis_unit':'recording',
        'serial_to_patient_mapping_available':False,
        'internal_claim':'recording-level provenance lock',
        'held_out_protection':'official CAUEEG no-overlap assignment preserved from P1-E02.2',
    },
    'manifest_csv':str(manifest_csv),
    'manifest_sha256':sha256_file(manifest_csv),
    'python':sys.version,
    'platform':platform.platform(),
    'status':status,
}
(OUTPUT_ROOT/'run_record.json').write_text(json.dumps(run_record, indent=2), encoding='utf-8')
print(json.dumps(run_record, indent=2))
if not checks['pass'].all():
    raise AssertionError(status)


## Takeaways

1. This notebook must begin with exactly 1,045 P1-E02.2 official no-overlap recordings.
2. The final artifact is a **recording-level** provenance manifest; it does not assert independently verified participant identities.
3. Every retained record keeps its original train, validation, or test assignment.
4. Cross-label or cross-split exact duplicates stop execution and require documented manual review.
5. All exclusions are recorded; raw EDF and event files remain unchanged.
6. Full-cohort age adjustment is intentionally absent. Any confound adjustment must later be learned only from training data.
7. **Next after a PASS:** P1-E04 event-defined non-photic eyes-closed interval eligibility and timing-reliability audit, using only `11_LOCKED_RECORDING_LEVEL_PROVENANCE_MANIFEST.csv`.

